<a href="https://colab.research.google.com/github/jhasankbharadwaj/carprice_prediction/blob/main/Dynamic_Pricing_Suggestion_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Neural Networks - Programming Assignment
## Comparing Linear Models and Multi-Layer Perceptrons

**Student Name:** T V M V C Jhasank Bharadwaj

**Student ID:** 2025AG05622  





In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

print('✓ Libraries imported successfully')


✓ Libraries imported successfully


In [3]:
data= pd.read_csv('/content/Retail-Supply-Chain-Sales-Dataset.csv', encoding='latin1')
data.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Retail-Supply-Chain-Sales-Dataset.csv'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
data.shape

In [ ]:
data.describe()


In [ ]:
data.info()


In [ ]:
col_name=data.columns


In [ ]:
for i in data.columns:
    print ("number of unique :{}\n{}\n uniquevalues \n{}".format(i,data[i].nunique(),data[i].unique()))
    print ("---------------------- \n")

*These columns are identifiers or near-unique attributes (high cardinality), so they don’t carry generalizable patterns—models tend to memorize them rather than learn relationships.
They introduce noise and overfitting, especially in ML/DL models, because each value behaves like a unique key instead of a meaningful feature.
Some (like Order ID, Customer ID, Product ID) can also cause data leakage or artificial correlations, while Country has no variance, making it statistically useless.*

In [ ]:
data = data.drop([
    'Row ID', 'Order ID', 'Customer ID', 'Customer Name',
    'Product ID', 'Product Name', 'Country', 'Postal Code'
], axis=1)

*Extracting the year from dates helps the model notice overall trends, like how things change from one year to another. But using only the year can miss important details, so adding features like month or day often helps the model learn better patterns.*

In [ ]:
data['Order Date'] = pd.to_datetime(data['Order Date'], dayfirst=True, errors='coerce')
data['Ship Date'] = pd.to_datetime(data['Ship Date'], dayfirst=True, errors='coerce')

# Extract features
data['order_year'] = data['Order Date'].dt.year
data['order_month'] = data['Order Date'].dt.month

# Derived feature
data['shipping_delay'] = (data['Ship Date'] - data['Order Date']).dt.days

# Drop original columns
data.drop(['Order Date', 'Ship Date'], axis=1, inplace=True)

In [ ]:
#as profit is highly corelated to sales we need to drop .

**Train test split**

In [ ]:
y = data['Sales']
X = data.drop(['Sales', 'Profit'], axis=1)



In [ ]:
X.describe(include='all')

In [ ]:
data_cat=X.select_dtypes(include=['object'])
data_numerical=X.select_dtypes(exclude=['object'])
print('data_cat columns',data_cat.columns,'\n')
print('data_numerical columns',data_numerical.columns)



```
# This is formatted as code
```

train test split


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)



# **3. Encode categorical variables**

*Different categorical features have different characteristics, so using a single encoding method for all can either lose information or create too many unnecessary features.
Applying multiple encoding techniques ensures each feature is represented efficiently while improving model performance and avoiding issues like dimensional explosion.*

In [ ]:
city_freq = X_train['City'].value_counts()
X_train['City'] = X_train['City'].map(city_freq)
X_test['City'] = X_test['City'].map(city_freq)

X_train['Returned'] = X_train['Returned'].map({'Yes': 1, 'No': 0})
X_test['Returned'] = X_test['Returned'].map({'Yes': 1, 'No': 0})

ohe_cols = [
    'Ship Mode', 'Segment', 'Region',
    'Category', 'Retail Sales People',
    'State', 'Sub-Category'
]

X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=ohe_cols, drop_first=True)

# 4. Align columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

**Model Training**

In [ ]:
class BaselineRegressionModel:
    """
    Baseline linear model with gradient descent
    Implement: Linear/Logistic/Softmax Regression
    """
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X, y):
        """
         Implement gradient descent training

        Steps:
        1. Initialize weights and bias
        2. For each iteration:
           a. Compute predictions (forward pass)
           b. Compute loss
           c. Compute gradients
           d. Update weights and bias
           e. Store loss in self.loss_history

        Must populate self.loss_history with loss at each iteration!
        """
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iterations):
            # 1. Forward pass
            linear_pred = np.dot(X, self.weights) + self.bias
            # 2. Compute loss
            loss = (1 / n_samples) * np.sum((linear_pred - y)**2)
            # 3. Compute gradients
            dw = (2 / n_samples) * np.dot(X.T, (linear_pred - y))
            db = (2 / n_samples) * np.sum(linear_pred - y)
            # 4. Update weights
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            # 5. Append loss
            self.loss_history.append(loss)
        return self

    def predict(self, X):
        """
         Implement prediction

        For regression: return linear_output
        For classification: return class probabilities or labels
        """
        return np.dot(X, self.weights) + self.bias



In [ ]:
class MLP:
    """
    Multi-Layer Perceptron implemented from scratch
    """
    def __init__(self, architecture, learning_rate=0.01, n_iterations=1000):
        """
        architecture: list [input_size, hidden1, hidden2, ..., output_size]
        Example: [15, 15, 8, 1] means:
            - 15 input features
            - Hidden layer 1: 15 neurons
            - Hidden layer 2: 8 neurons
            - Output layer: 1 neuron
        """
        self.architecture = architecture
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.parameters = {}
        self.loss_history = []
        self.cache = {}

    def initialize_parameters(self):
        """
        For each layer l:
        - W[l]: weight matrix of shape (n[l], n[l-1])
        - b[l]: bias vector of shape (n[l], 1)

        Store in self.parameters dictionary
        """
        np.random.seed(42)

        for l in range(1, len(self.architecture)):
            #  Initialize weights and biases
            self.parameters[f'W{l}'] = np.random.randn(self.architecture[l-1], self.architecture[l]) * 0.01

            self.parameters[f'b{l}'] = np.zeros((1, self.architecture[l]))

    def relu(self, Z):
        """ReLU activation function"""
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        """ReLU derivative"""
        return (Z > 0).astype(float)

    def sigmoid(self, Z):
        """Sigmoid activation (for binary classification output)"""
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

    def forward_propagation(self, X):
        """
        For each layer:
        1. Z[l] = W[l] @ A[l-1] + b[l]
        2. A[l] = activation(Z[l])

        Store Z and A in self.cache for backpropagation
        Return final activation A[L]
        """
        self.cache['A0'] = X
        A = X

        for l in range(1, len(self.architecture)):
            W_l = self.parameters[f'W{l}']
            b_l = self.parameters[f'b{l}']

            Z_l = np.dot(A, W_l) + b_l
            self.cache[f'Z{l}'] = Z_l

            if l < len(self.architecture) - 1:
                A = self.relu(Z_l)
            else:
                A = Z_l
            self.cache[f'A{l}'] = A

        return A

    def backward_propagation(self, X, y):
        """
        Starting from output layer, compute:
        1. dZ[l] for each layer
        2. dW[l] = dZ[l] @ A[l-1].T / m
        3. db[l] = sum(dZ[l]) / m

        Return dictionary of gradients
        """
        m = X.shape[0]
        grads = {}

        y_reshaped = y.reshape(-1, 1)
        num_layers = len(self.architecture) - 1
        A_L = self.cache[f'A{num_layers}']
        dZ = A_L - y_reshaped

        # Gradients for the output layer's weights and biases
        grads[f'dW{num_layers}'] = (1 / m) * np.dot(self.cache[f'A{num_layers - 1}'].T, dZ)
        grads[f'db{num_layers}'] = (1 / m) * np.sum(dZ, axis=0, keepdims=True)

        # Propagate gradients backwards through hidden layers
        for l in range(num_layers - 1, 0, -1):
            W_l_plus_1 = self.parameters[f'W{l+1}']  # Weights of the next layer (in forward direction)
            dA = np.dot(dZ, W_l_plus_1.T)
            dZ = dA * self.relu_derivative(self.cache[f'Z{l}'])

            # Gradients for current layer's weights and biases
            grads[f'dW{l}'] = (1 / m) * np.dot(self.cache[f'A{l-1}'].T, dZ)
            grads[f'db{l}'] = (1 / m) * np.sum(dZ, axis=0, keepdims=True)

        return grads

    def update_parameters(self, grads):
        """
        For each layer:
        W[l] = W[l] - learning_rate * dW[l]
        b[l] = b[l] - learning_rate * db[l]
        """
        num_layers = len(self.architecture) - 1
        for l in range(1, num_layers + 1):
            self.parameters[f'W{l}'] -= self.lr * grads[f'dW{l}']
            self.parameters[f'b{l}'] -= self.lr * grads[f'db{l}']

    def compute_loss(self, y_pred, y_true):

        m = y_true.shape[0]
        loss = (1 / (2 * m)) * np.sum((y_pred - y_true.reshape(-1, 1)) ** 2)
        return loss

    def fit(self, X, y):

        self.initialize_parameters()

        for i in range(self.n_iterations):
            y_pred = self.forward_propagation(X)
            loss = self.compute_loss(y_pred, y)
            grads = self.backward_propagation(X, y)
            self.update_parameters(grads)
            self.loss_history.append(loss)

        return self

    def predict(self, X):

        y_pred = self.forward_propagation(X)

        return y_pred.flatten()

In [ ]:
import time

print("Training MLP...")



Y_train = Y_train.values.reshape(-1, 1)
Y_test = Y_test.values.reshape(-1, 1)

# Start timer BEFORE training
mlp_start_time = time.time()

# Define model
mlp_architecture = [X_train.shape[1], 32, 16, 8, 1]
mlp_model = MLP(
    architecture=mlp_architecture,
    learning_rate=0.001,   # better than 0.01
    n_iterations=1000
)

# Train
mlp_model.fit(X_train, Y_train)

# Predictions
mlp_predictions = mlp_model.predict(X_test)

# End timer
mlp_training_time = time.time() - mlp_start_time

# Output
print(f"✓ MLP training completed in {mlp_training_time:.2f}s")
print(f"✓ Loss decreased from {mlp_model.loss_history[0]:.4f} to {mlp_model.loss_history[-1]:.4f}")